# 01 Python + 监督学习基础

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanteen/COF-ML-Tutorial/blob/main/notebooks/01_python_ml_basics.ipynb)

## Learning objectives
完成本节后，你应该能够解释 feature、target、train/test split、overfitting，并独立完成一个材料性质回归 baseline。

流程：`DataFrame → split → preprocessing → model → MAE/RMSE/R² → parity plot`。

## 1. 先理解问题，而不是先选模型

监督学习假设我们已经有一组材料 $X$ 及其已知性质 $y$。模型学习映射 $f(X)\rightarrow y$。材料机器学习最常见的问题不是模型不够复杂，而是数据量、数据质量、表示方式和验证方案不合理。

**Feature** 是模型输入；**target/label** 是要预测的量；**training set** 用于拟合；**test set** 只用于最终评估泛化。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rng = np.random.default_rng(42)
n = 300
df = pd.DataFrame({
    'density': rng.uniform(0.3, 1.5, n),
    'pore_diameter': rng.uniform(6, 35, n),
    'void_fraction': rng.uniform(0.25, 0.85, n),
    'N_fraction': rng.uniform(0.0, 0.20, n),
    'O_fraction': rng.uniform(0.0, 0.20, n),
})
df['target'] = (2*df['void_fraction'] + 0.08*df['pore_diameter'] + 3*df['N_fraction'] - 0.6*df['density'] + rng.normal(0,0.25,n))
df.head()

## 2. 为什么一定要划分测试集？
模型在训练数据上的误差回答的是“记住/拟合训练样本的程度”，而我们真正关心的是它对未见材料的泛化能力。后面的 COF 章节还会进一步说明：**随机划分本身也可能过于乐观**。

In [ ]:
X = df.drop(columns='target')
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Ridge': make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    'RandomForest': RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f'{name:12s} MAE={mean_absolute_error(y_test,pred):.3f}  RMSE={mean_squared_error(y_test,pred)**0.5:.3f}  R2={r2_score(y_test,pred):.3f}')

## 3. 三个常用指标
- **MAE**：平均绝对误差，最容易解释。
- **RMSE**：对大误差惩罚更强。
- **R²**：描述相对基线的解释程度，但不能替代带物理单位的误差。

材料论文中不要只报告 R²；至少同时给出 MAE/RMSE，并说明数据划分方法。

In [ ]:
model = models['RandomForest']
pred = model.predict(X_test)
plt.figure(figsize=(5,5))
plt.scatter(y_test, pred, alpha=0.7)
lims=[min(y_test.min(),pred.min()),max(y_test.max(),pred.max())]
plt.plot(lims,lims,'--')
plt.xlabel('True'); plt.ylabel('Predicted'); plt.title('Parity plot'); plt.show()

## Exercises
1. 将 `test_size` 改成 0.1 和 0.3，记录指标变化。
2. 删除 `pore_diameter` 后重新训练，解释变化。
3. 比较 Ridge 与 Random Forest：哪一个更适合非线性关系？
4. 用自己的话解释 overfitting 和 data leakage。
5. **思考题：** 如果同一 COF family 的高度相似结构同时出现在 train 和 test，测试结果还可信吗？

### Take-home message
可靠的数据划分和评价方法通常比盲目增加模型复杂度更重要。下一节开始把这些概念连接到真实周期材料结构。